c_search_w_dspy.ipynb

## most recent version needs testings - see mmarkdown explanation

In [ ]:
import dspy
from dspy.teleprompt import BootstrapFewShot
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import boto3
import json
import faiss
import numpy as np
import io
from typing import List, Dict

# Configuration Dictionary
CONFIG = {
    "bucket_name": "sagemaker-us-east-1-188494237500",
    "csv_folder": "csv_metadata",
    "index_folder": "faiss_index",
    "embedding_models": {
        "amazon.titan-embed-text-v1": 1536,
        "anthropic.claude-3-sonnet-20240229-v1:0": 8192,
        "meta.llama3-70b-instruct-v1:0": 8192
    },
    "nprobe": 16
}

# Initialize AWS Clients
s3 = boto3.client('s3')
bedrock = boto3.client('bedrock-runtime')

# UI Widgets
embedding_model_widget = widgets.Dropdown(
    options=list(CONFIG["embedding_models"].keys()),
    description="Embedding Model:"
)
query_widget = widgets.Textarea(description="Query:", placeholder="Enter your search query...")
search_button = widgets.Button(description="Search")
results_output = widgets.Output()

display(embedding_model_widget, query_widget, search_button, results_output)

# Load FAISS indexes
faiss_indexes = {model: None for model in CONFIG["embedding_models"]}

def load_faiss_index(model_name: str) -> faiss.Index:
    """Loads FAISS index for a given model."""
    embedding_dim = CONFIG["embedding_models"][model_name]
    index_filename = f"faiss_index_{model_name.replace(':', '_')}_{embedding_dim}.faiss"
    index_path = f"/tmp/{index_filename}"

    try:
        s3.download_file(CONFIG["bucket_name"], f"{CONFIG['index_folder']}/{index_filename}", index_path)
        index = faiss.read_index(index_path)
        print(f"✅ FAISS index loaded for {model_name}")
        return index
    except s3.exceptions.ClientError:
        print(f"⚠️ No FAISS index found for {model_name}. Skipping.")
        return None

for model in CONFIG["embedding_models"]:
    faiss_indexes[model] = load_faiss_index(model)

# DSPy Signatures
class RationaleSignature(dspy.Signature):
    """Generate rationale for a query based on retrieved documents."""
    query: str = dspy.InputField()
    sources: list = dspy.InputField()
    rationale: str = dspy.OutputField()

class ChainOfThoughtQASignature(dspy.Signature):
    """Answer questions using Chain of Thought."""
    query: str = dspy.InputField()
    context: str = dspy.InputField()
    reasoning_steps: str = dspy.OutputField()
    final_answer: str = dspy.OutputField()

# DSPy Modules
class RationaleModel(dspy.Module):
    def __init__(self):
        super().__init__()

    @dspy.predict(RationaleSignature)
    def __call__(self, query, sources):
        return RationaleSignature(query=query, sources=sources)

class ChainOfThoughtQAModel(dspy.Module):
    def __init__(self):
        super().__init__()

    @dspy.predict(ChainOfThoughtQASignature)
    def __call__(self, query, context):
        return ChainOfThoughtQASignature(query=query, context=context)

# **🔹 New: DSPy Compiler Using `BootstrapFewShot`**
teleprompter = BootstrapFewShot(metric=None)  # No metric for now
compiled_qa = teleprompter.compile(ChainOfThoughtQAModel())

# Query FAISS Index
def query_faiss(query_embedding: np.ndarray, model_name: str, top_k: int = 5):
    """Queries FAISS index and retrieves top results."""
    index = faiss_indexes.get(model_name)
    if index is None:
        print(f"⚠️ No FAISS index available for {model_name}. Skipping search.")
        return {}

    index.nprobe = CONFIG["nprobe"]
    distances, indices = index.search(query_embedding, top_k)
    return {"indices": indices[0].tolist(), "distances": distances[0].tolist()}

# Retrieve Query Embedding
def get_query_embedding(query_text: str, model_name: str) -> np.ndarray:
    """Ensures query embedding matches the model-specific FAISS index."""
    if model_name not in CONFIG["embedding_models"]:
        print(f"⚠️ Model {model_name} not found in configuration.")
        return None

    embedding_dim = CONFIG["embedding_models"][model_name]
    index = faiss_indexes.get(model_name)

    if index is None:
        print(f"⚠️ No FAISS index found for {model_name}.")
        return None

    query_embedding = np.zeros((1, embedding_dim), dtype=np.float32)
    
    if index.ntotal > 0:
        _, query_embedding_index = index.search(query_embedding, 1)
        return query_embedding_index.astype(np.float32)

    print("⚠️ No stored embeddings found in FAISS index.")
    return None

# Perform Search and Apply DSPy Reasoning
def perform_search(_):
    """Executes FAISS search and applies DSPy reasoning."""
    query_text = query_widget.value
    model_name = embedding_model_widget.value

    query_embedding = get_query_embedding(query_text, model_name)
    if query_embedding is None:
        with results_output:
            results_output.clear_output()
            print(f"⚠️ Could not retrieve embedding for query: {query_text}")
        return

    search_results = query_faiss(query_embedding, model_name)

    if not search_results:
        with results_output:
            results_output.clear_output()
            print("⚠️ No results found in FAISS index.")
        return

    # **New: Use DSPy Prompter for Enhanced Rationale**
    retrieved_texts = [f"Document {i}: [Relevant Text]" for i in search_results["indices"]]
    rationale_model = RationaleModel()
    rationale_output = rationale_model(query=query_text, sources=retrieved_texts)

    qa_response = compiled_qa(query=query_text, context="\n".join(retrieved_texts))

    with results_output:
        results_output.clear_output()
        print(f"🔎 Query: {query_text}")
        print(f"📄 Retrieved Documents: {retrieved_texts}")
        print(f"📝 Rationale: {rationale_output.rationale}")
        print(f"✅ Answer: {qa_response.final_answer}")

search_button.on_click(perform_search)

# Display UI
display(widgets.HTML("<h2 style='color: navy; background-color: lightgray; padding: 10px; border-radius: 8px;'>Multi-Model FAISS + DSPy Search</h2>"))
